# OTTER quickstart

Load the released canonical coupling, query one mouse parcel and aggregate a mouse region. Download the release bundle with python scripts/fetch_data.py before running this notebook.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from otter.data import load_cached, load_pi, pi_provenance

mouse, _ = load_cached('mouse', cache_dir=ROOT / 'outputs/anndata')
human, _ = load_cached('human', cache_dir=ROOT / 'outputs/anndata')
pi = load_pi()
print(pi.shape)
print(pi_provenance())

## Query one parcel

A coupling entry is a transport weight. Normalise the selected mouse row before reading it as a distribution over human targets.

In [ ]:
mouse_idx = 1234
weights = pi[mouse_idx] / pi[mouse_idx].sum()
top = np.argsort(weights)[::-1][:10]

result = human.var.iloc[top][['region', 'subregion', 'x', 'y', 'z']].copy()
result['weight'] = weights[top]
result

## Aggregate a mouse region

Region-level queries are generally more robust than parcel-exact assignments. Sum the rows in the source region and then normalise.

In [ ]:
region_name = mouse.var['region'].value_counts().index[0]
source = np.flatnonzero(mouse.var['region'].to_numpy() == region_name)
region_weights = pi[source].sum(axis=0)
region_weights /= region_weights.sum()
top = np.argsort(region_weights)[::-1][:10]

result = human.var.iloc[top][['region', 'subregion', 'x', 'y', 'z']].copy()
result['weight'] = region_weights[top]
print(f'{region_name}: {len(source)} mouse parcels')
result

## Interface metadata

The explorer bundle records anchor membership, benchmark-region membership and internal stability summaries. These labels control display granularity; they are not confidence probabilities or additional validation results.